<a href="https://colab.research.google.com/github/LIKITHKOLLURU/AMLTA-EXPS/blob/main/SKILL_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense

In [2]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
word_index = imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
index_to_word = {index + 3: word for word, index in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[1] = "<START>"
index_to_word[2] = "<UNK>"
index_to_word[3] = "<UNUSED>"

In [5]:
def decode_review(encoded_review):
  return ' '.join([index_to_word.get(i, '?') for i in encoded_review])

In [6]:
for i in range(1):
  print(f"Review {i+1}:")
  print(decode_review(x_train[i]))
  print("Label:", "Positive" if y_train[i] == 1 else "Negative")
  print("-" * 100)

Review 1:
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be prais

In [7]:
x_train = pad_sequences(x_train, maxlen=200)
x_test = pad_sequences(x_test, maxlen=200)
 # Build RNN model
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=32, input_length=200))
model.add(SimpleRNN(units=32))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.2)
loss, accuracy = model.evaluate(x_test, y_test)
print(f'Test Accuracy: {accuracy:.2f}')

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - accuracy: 0.5513 - loss: 0.6783 - val_accuracy: 0.7118 - val_loss: 0.5608
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 48ms/step - accuracy: 0.7734 - loss: 0.4830 - val_accuracy: 0.7936 - val_loss: 0.4590
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step - accuracy: 0.8899 - loss: 0.2840 - val_accuracy: 0.6330 - val_loss: 0.6851
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step - accuracy: 0.9005 - loss: 0.2648 - val_accuracy: 0.7486 - val_loss: 0.6184
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 48ms/step - accuracy: 0.9718 - loss: 0.0895 - val_accuracy: 0.7952 - val_loss: 0.6033
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7860 - loss: 0.6187
Test Accuracy: 0.79


In [9]:
predictions = model.predict(x_test)
predicted_labels = [1 if p >= 0.5 else 0 for p in predictions]
 # Function to decode a review without <PAD> and special tokens
def decode_review_clean(encoded_review):
  words = [index_to_word.get(i, '?') for i in encoded_review if i > 3]
  return ' '.join(words)
 # Show first 5 test reviews with predictions
for i in range(5):
  print(f"\nTest Review {i+1}:")
  print(decode_review_clean(x_test[i]))
  print("Predicted:", "Positive" if predicted_labels[i] == 1 else "Negative")
  print("Actual:   ", "Positive" if y_test[i] == 1 else "Negative")
  print("-" * 100)

782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step

Test Review 1:
please give this one a miss br br and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite so all you madison fans give this a miss
Predicted: Negative
Actual:    Negative
----------------------------------------------------------------------------------------------------

Test Review 2:
psychological it's very interesting that robert altman directed this considering the style and structure of his other films still the trademark altman audio style is evident here and there i think what really makes this film work is the brilliant performance by sandy dennis it's definitely one of her darker characters but she plays it so perfectly and convincingly that it's scary michael burns does a good job as the mute young man regular altman player micha

In [10]:
print(f"Total training reviews: {len(x_train)}")
print(f"Total test reviews: {len(x_test)}")
print(f"Total reviews: {len(x_train) + len(x_test)}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")
print("Class labels: 0 (Negative), 1 (Positive)")

Total training reviews: 25000
Total test reviews: 25000
Total reviews: 50000
Training labels shape: (25000,)
Test labels shape: (25000,)
Class labels: 0 (Negative), 1 (Positive)


In [11]:
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(y_test, predicted_labels)

# Print the confusion matrix with labels
print("Confusion Matrix:")
print(f"True Negatives (TN): {cm[0, 0]}")
print(f"False Positives (FP): {cm[0, 1]}")
print(f"False Negatives (FN): {cm[1, 0]}")
print(f"True Positives (TP): {cm[1, 1]}")

# Print the confusion matrix in matrix form
print("\nConfusion Matrix (Matrix Form):")
print(cm)

Confusion Matrix:
True Negatives (TN): 10202
False Positives (FP): 2298
False Negatives (FN): 2943
True Positives (TP): 9557

Confusion Matrix (Matrix Form):
[[10202  2298]
 [ 2943  9557]]


In [12]:
from sklearn.metrics import classification_report
print("Classification Report:\n")
print(classification_report(y_test, predicted_labels, target_names=['Negative', 'Positive']))

Classification Report:

              precision    recall  f1-score   support

    Negative       0.78      0.82      0.80     12500
    Positive       0.81      0.76      0.78     12500

    accuracy                           0.79     25000
   macro avg       0.79      0.79      0.79     25000
weighted avg       0.79      0.79      0.79     25000

